# Notebook 2: CNN 역할 분류기

## 목표
EfficientNet-B3를 fine-tuning해서 슬라이드 역할을 분류하는 모델을 학습.

| 클래스 | 역할 |
|---|---|
| 0 | 표지 |
| 1 | 섹션헤더 |
| 2 | 본문 |
| 3 | 도표/시각자료 |
| 4 | 마무리 |

## 예상 소요 시간 (A100)
- 정규화 통계 계산: 5분
- Stage 1 학습 (5 epoch): 10분
- Stage 2 학습 (15 epoch): 30~40분

## 0. Drive 마운트 및 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/dadeum_ml'
LABELS_DIR = f'{BASE_DIR}/labels'
MODELS_DIR = f'{BASE_DIR}/models'

import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f}GB')

## 1. 패키지 설치

In [ ]:
!pip install -q timm torchmetrics umap-learn
print('설치 완료')

## 2. 데이터 로더

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROLE_NAMES = ['표지', '섹션헤더', '본문', '도표/시각자료', '마무리']
NUM_CLASSES = 5
IMG_SIZE = 224


class SlideRoleDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df[df['image_path'].apply(lambda p: Path(p).exists())].reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['image_path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        label = int(row['weak_label'])
        return img, label


# 데이터 로드 및 분할
df = pd.read_csv(f'{LABELS_DIR}/weak_labels.csv')
print(f'전체 슬라이드: {len(df)}장')
print(df['role_name'].value_counts())

deck_ids = df['deck_id'].unique()
np.random.seed(42)
np.random.shuffle(deck_ids)

split = int(len(deck_ids) * 0.85)
train_decks = set(deck_ids[:split])
val_decks = set(deck_ids[split:])

train_df = df[df['deck_id'].isin(train_decks)].reset_index(drop=True)
val_df = df[df['deck_id'].isin(val_decks)].reset_index(drop=True)

print(f'\n학습 슬라이드: {len(train_df)}장 ({len(train_decks)}개 덱)')
print(f'검증 슬라이드: {len(val_df)}장 ({len(val_decks)}개 덱)')

## 2-A. 슬라이드 데이터 정규화 통계 계산

ImageNet 통계 대신 실제 슬라이드 픽셀 분포로 정규화.

In [ ]:
from tqdm import tqdm as tqdm_plain

sample_df = train_df.sample(min(5000, len(train_df)), random_state=42)
raw_transform = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor()])

mean_sum = torch.zeros(3)
sq_sum   = torch.zeros(3)
n = 0
for _, row in tqdm_plain(sample_df.iterrows(), total=len(sample_df), desc='정규화 통계 계산'):
    if not Path(row['image_path']).exists():
        continue
    t = raw_transform(Image.open(row['image_path']).convert('RGB'))
    mean_sum += t.mean(dim=(1, 2))
    sq_sum   += (t ** 2).mean(dim=(1, 2))
    n += 1

SLIDE_MEAN = (mean_sum / n).tolist()
SLIDE_STD  = ((sq_sum / n - torch.tensor(SLIDE_MEAN) ** 2).sqrt()).tolist()
print(f'슬라이드 mean: {[round(v, 4) for v in SLIDE_MEAN]}')
print(f'슬라이드 std:  {[round(v, 4) for v in SLIDE_STD]}')

import json
with open(f'{MODELS_DIR}/slide_norm.json', 'w') as f:
    json.dump({'mean': SLIDE_MEAN, 'std': SLIDE_STD}, f)
print(f'정규화 통계 저장: {MODELS_DIR}/slide_norm.json')

In [ ]:
BATCH_SIZE = 128  # A100 40GB 기준
NUM_WORKERS = 4

# 슬라이드에 맞는 증강 (HorizontalFlip 제거 — 슬라이드는 16:9 고정 방향)
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=SLIDE_MEAN, std=SLIDE_STD),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=SLIDE_MEAN, std=SLIDE_STD),
])

train_dataset = SlideRoleDataset(train_df, train_transform)
val_dataset = SlideRoleDataset(val_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'학습 배치 수: {len(train_loader)}')
print(f'검증 배치 수: {len(val_loader)}')

## 3. EfficientNet-B3 모델 설계

In [ ]:
import timm
import torch.nn as nn


class SlideRoleClassifier(nn.Module):
    def __init__(self, num_classes: int = 5, dropout: float = 0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'efficientnet_b3',
            pretrained=True,
            num_classes=0,
            global_pool='avg',
        )
        feat_dim = self.backbone.num_features  # 1536

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits

    def extract_features(self, x):
        """임베딩만 반환 — Isolation Forest 입력으로 사용"""
        return self.backbone(x)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SlideRoleClassifier(num_classes=NUM_CLASSES).to(device)

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'전체 파라미터: {total:,}')
print(f'학습 가능 파라미터: {trainable:,}')

## 4. 학습 (2단계 Backbone Freeze Fine-tuning)

- Stage 1: backbone 동결, head warm-up (5 epoch)
- Stage 2: backbone 해제, 차등 학습률 + early stopping (patience=5)

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler
from torchmetrics import Accuracy
import json as _json
import time

# 클래스 가중치 — class_weights.json 우선 로드 (step1 저장), 없으면 train_df에서 계산
json_path = Path(f'{LABELS_DIR}/class_weights.json')
if json_path.exists():
    with open(json_path) as f:
        raw = _json.load(f)
    class_weights_list = [raw[str(i)] for i in range(NUM_CLASSES)]
    class_weights = torch.tensor(class_weights_list, dtype=torch.float32).to(device)
    print('class_weights.json에서 가중치 로드')
else:
    counts = train_df['weak_label'].value_counts().sort_index().values
    class_weights = torch.tensor(1.0 / counts, dtype=torch.float32).to(device)
    print('train_df에서 클래스 가중치 계산 (fallback)')

class_weights = class_weights / class_weights.sum() * NUM_CLASSES

# 약한 라벨 오류율 ~15~20% 보정용 label_smoothing
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
scaler = GradScaler('cuda')
accuracy = Accuracy(task='multiclass', num_classes=NUM_CLASSES).to(device)

best_val_acc = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}


def run_epoch(loader, is_train, optimizer=None):
    """한 epoch 실행. is_train=True면 학습, False면 검증."""
    if is_train:
        model.train()
    else:
        model.eval()
    total_loss = 0.0
    accuracy.reset()

    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if is_train:
            optimizer.zero_grad()
            with autocast('cuda'):
                logits = model(imgs)
                loss   = criterion(logits, labels)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                logits = model(imgs)
                loss   = criterion(logits, labels)
        total_loss += loss.item()
        accuracy.update(logits, labels)

    return total_loss / len(loader), accuracy.compute().item()


# --- Stage 1: backbone 동결, head만 warm-up (5 epoch) ---
print('=== Stage 1: backbone 동결 (5 epoch) ===')
for param in model.backbone.parameters():
    param.requires_grad = False

optimizer_s1 = AdamW(model.classifier.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler_s1 = CosineAnnealingLR(optimizer_s1, T_max=5, eta_min=1e-5)

for epoch in range(5):
    t0 = time.time()
    train_loss, _ = run_epoch(train_loader, is_train=True, optimizer=optimizer_s1)
    val_loss, val_acc = run_epoch(val_loader, is_train=False)
    scheduler_s1.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'S1 Epoch {epoch+1:2d}/5 | train={train_loss:.4f} | val={val_loss:.4f} | acc={val_acc:.4f} | {time.time()-t0:.1f}s')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(), 'val_acc': val_acc},
                   f'{MODELS_DIR}/role_classifier_best.pt')

# --- Stage 2: backbone 해제, 차등 학습률 + early stopping ---
print('\n=== Stage 2: 차등 학습률 (최대 15 epoch) ===')
for param in model.backbone.parameters():
    param.requires_grad = True

optimizer_s2 = AdamW([
    {'params': model.backbone.parameters(),   'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4},
], weight_decay=1e-2)
scheduler_s2 = CosineAnnealingLR(optimizer_s2, T_max=15, eta_min=1e-6)

patience = 5
no_improve = 0

for epoch in range(15):
    t0 = time.time()
    train_loss, _ = run_epoch(train_loader, is_train=True, optimizer=optimizer_s2)
    val_loss, val_acc = run_epoch(val_loader, is_train=False)
    scheduler_s2.step()
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f'S2 Epoch {epoch+1:2d}/15 | train={train_loss:.4f} | val={val_loss:.4f} | acc={val_acc:.4f} | {time.time()-t0:.1f}s')
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        no_improve = 0
        torch.save({'epoch': epoch + 5, 'model_state_dict': model.state_dict(), 'val_acc': val_acc},
                   f'{MODELS_DIR}/role_classifier_best.pt')
        print(f'  → 최고 모델 저장 (val_acc={val_acc:.4f})')
    else:
        no_improve += 1
        if no_improve >= patience:
            print(f'  Early stopping (patience={patience})')
            break

print(f'\n학습 완료. 최고 검증 정확도: {best_val_acc:.4f}')

In [ ]:
# 학습 곡선 시각화
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.axvline(5, color='gray', linestyle=':', label='Stage1→2')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve')
ax1.legend()

ax2.plot(history['val_acc'], label='Val Accuracy', color='green')
ax2.axvline(5, color='gray', linestyle=':', label='Stage1→2')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Validation Accuracy')
ax2.legend()

plt.tight_layout()
plt.savefig(f'{MODELS_DIR}/training_curves.png', dpi=100)
plt.show()

## 5. CNN 임베딩 추출 + PCA 차원 축소

1536차원 임베딩 → PCA 95% 분산 보존. Isolation Forest는 고차원에서 차원의 저주로 동작하지 않으므로 반드시 필요.

In [ ]:
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import pickle

# 최고 모델 로드
checkpoint = torch.load(f'{MODELS_DIR}/role_classifier_best.pt', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print(f'모델 로드 완료 (val_acc={checkpoint["val_acc"]:.4f})')

# 전체 데이터셋 임베딩 추출
full_dataset = SlideRoleDataset(df, val_transform)
full_loader = DataLoader(full_dataset, batch_size=256,
                         shuffle=False, num_workers=4, pin_memory=True)

all_embeddings = []
all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in tqdm(full_loader, desc='임베딩 추출'):
        imgs = imgs.to(device)
        embeddings = model.extract_features(imgs)
        logits = model.classifier(embeddings)
        preds = logits.argmax(dim=1)

        all_embeddings.append(embeddings.cpu().numpy())
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.numpy())

embeddings_np = np.concatenate(all_embeddings, axis=0)  # (N, 1536)
preds_np = np.concatenate(all_preds, axis=0)
labels_np = np.concatenate(all_labels, axis=0)

# PCA 차원 축소 (GradScaler와 구분하기 위해 pca_scaler 변수명 사용)
pca_scaler = StandardScaler()
embeddings_scaled = pca_scaler.fit_transform(embeddings_np)

pca = PCA(n_components=0.95, svd_solver='full', random_state=42)
embeddings_pca = pca.fit_transform(embeddings_scaled)

print(f'PCA: {embeddings_np.shape[1]}차원 → {embeddings_pca.shape[1]}차원')
print(f'보존 분산: {pca.explained_variance_ratio_.sum():.3%}')

with open(f'{MODELS_DIR}/pca_model.pkl', 'wb') as f:
    pickle.dump({'scaler': pca_scaler, 'pca': pca}, f)

np.save(f'{LABELS_DIR}/embeddings.npy', embeddings_np)
np.save(f'{LABELS_DIR}/embeddings_pca.npy', embeddings_pca)

# CNN 예측 역할을 df에 추가
valid_df = full_dataset.df.copy()
valid_df['pred_label'] = preds_np
valid_df['pred_role'] = [ROLE_NAMES[p] for p in preds_np]
valid_df.to_csv(f'{LABELS_DIR}/labeled_with_preds.csv', index=False)

print(f'임베딩 저장: {embeddings_np.shape} → {LABELS_DIR}/embeddings.npy')
print(f'PCA 임베딩: {embeddings_pca.shape} → {LABELS_DIR}/embeddings_pca.npy')
print(f'PCA 모델: {MODELS_DIR}/pca_model.pkl')

acc = (preds_np == labels_np).mean()
print(f'\n전체 정확도: {acc:.4f}')

from sklearn.metrics import classification_report
print('\n분류 리포트:')
print(classification_report(labels_np, preds_np, target_names=ROLE_NAMES))

## 6. UMAP 임베딩 시각화

임베딩이 역할별로 클러스터링됐는지 확인.

In [ ]:
import umap

SAMPLE_N   = 3000
sample_idx = np.random.choice(len(embeddings_np), min(SAMPLE_N, len(embeddings_np)), replace=False)
emb_2d     = umap.UMAP(n_components=2, random_state=42).fit_transform(embeddings_np[sample_idx])

fig, ax = plt.subplots(figsize=(10, 8))
for role_id in range(NUM_CLASSES):
    mask = labels_np[sample_idx] == role_id
    ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1], label=ROLE_NAMES[role_id], alpha=0.6, s=10)

ax.legend(markerscale=3)
ax.set_title('EfficientNet-B3 임베딩 UMAP (역할별 색상)')
ax.axis('off')
plt.savefig(f'{MODELS_DIR}/embedding_umap.png', dpi=120)
plt.show()
print('UMAP 시각화 저장 완료')

In [ ]:
print('=== Notebook 2 완료 ===')
print(f'role_classifier_best.pt 저장 완료')
print(f'embeddings.npy 저장 완료: {embeddings_np.shape}')
print(f'embeddings_pca.npy 저장 완료: {embeddings_pca.shape}')
print(f'pca_model.pkl 저장 완료')
print(f'slide_norm.json 저장 완료')
print(f'최고 검증 정확도: {best_val_acc:.4f}')
print('\nNotebook 3 (HMM 학습)으로 이동하세요.')